In [1]:
# ============================================================
# NOTEBOOK 10 — AGENTE DE COBRANZA: RAG CON CHROMADB
# ============================================================
# Propósito  : Construir el sistema RAG del Agente de Cobranza.
#              Combina ChromaDB (vector store) + sentence-
#              transformers (embeddings) + Claude API (LLM)
#              para generar estrategias de cobranza
#              personalizadas por cliente.
#
# Bloques:
#   Bloque 1 — Crear documentos de conocimiento de cobranza
#   Bloque 2 — Construir vector store con ChromaDB
#   Bloque 3 — Función de consulta RAG
#   Bloque 4 — Integración con Claude API
#   Bloque 5 — Pruebas con clientes reales del EWS
#
# Autor      : Marín Serrato Barrios
# Fecha      : Mayo 2026
# ============================================================

# ── Librerías generales ──────────────────────────────────────
import os
import json
import joblib
import warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

# ── ChromaDB — vector store local ───────────────────────────
# chromadb almacena y busca vectores de texto en disco
# Es gratuito, local y no requiere conexión a internet
import chromadb
from chromadb.config import Settings

# ── Sentence Transformers — embeddings ───────────────────────
# Convierte texto en vectores numéricos que capturan
# el significado semántico del texto
from sentence_transformers import SentenceTransformer

# ── Anthropic — Claude API ───────────────────────────────────
import anthropic

# ── Rutas del proyecto ───────────────────────────────────────
BASE        = os.path.abspath('..')
MODELS_PATH = os.path.join(BASE, 'dashboard', 'models')
DATA_PATH   = os.path.join(BASE, 'data', 'processed')

# Carpeta donde guardaremos la base de conocimiento
KNOWLEDGE_PATH = os.path.join(BASE, 'src', 'collections')
os.makedirs(KNOWLEDGE_PATH, exist_ok=True)

# Carpeta donde ChromaDB guardará los vectores en disco
CHROMA_PATH = os.path.join(BASE, 'src', 'collections',
                            'chroma_db')
os.makedirs(CHROMA_PATH, exist_ok=True)

print('✅ Librerías cargadas')
print(f'   Base del proyecto  : {BASE}')
print(f'   Modelos            : {MODELS_PATH}')
print(f'   Base de conocimiento: {KNOWLEDGE_PATH}')
print(f'   ChromaDB           : {CHROMA_PATH}')

✅ Librerías cargadas
   Base del proyecto  : C:\Users\Marin\Documents\PROYECTO ML_OPS\credit-risk-scoring-ml
   Modelos            : C:\Users\Marin\Documents\PROYECTO ML_OPS\credit-risk-scoring-ml\dashboard\models
   Base de conocimiento: C:\Users\Marin\Documents\PROYECTO ML_OPS\credit-risk-scoring-ml\src\collections
   ChromaDB           : C:\Users\Marin\Documents\PROYECTO ML_OPS\credit-risk-scoring-ml\src\collections\chroma_db


In [3]:
# ============================================================
# CELDA 3 — CREAR DOCUMENTOS REALES (.docx y .xlsx)
# ============================================================
# Objetivo: generar los documentos de conocimiento en formatos
# reales (Word y Excel) que el RAG cargará e indexará.
#
# En producción estos archivos serían reemplazados por los
# documentos oficiales de la institución sin tocar el código.
#
# Estructura:
# src/collections/documentos/
#   01_estrategia_preventiva.docx
#   02_estrategia_temprana.docx
#   03_estrategia_media.docx
#   04_estrategia_tardia.docx
#   05_marco_legal_condusef.docx
#   06_perfiles_cliente.docx
#   07_clasificacion_cnbv.xlsx
# ============================================================

from docx import Document
from docx.shared import Pt, RGBColor, Inches
from docx.enum.text import WD_ALIGN_PARAGRAPH
import pandas as pd
import os

# Carpeta donde guardaremos los documentos
DOCS_PATH = os.path.join(KNOWLEDGE_PATH, 'documentos')
os.makedirs(DOCS_PATH, exist_ok=True)

# ── Función auxiliar para crear documentos Word ──────────────
def crear_docx(titulo, secciones, nombre_archivo):
    """
    Crea un documento Word con título y secciones.
    secciones: lista de dicts con 'subtitulo' y 'contenido'
    """
    doc = Document()

    # Título principal
    titulo_par = doc.add_heading(titulo, level=1)
    titulo_par.alignment = WD_ALIGN_PARAGRAPH.CENTER

    # Línea separadora
    doc.add_paragraph('─' * 60)

    # Secciones
    for seccion in secciones:
        # Subtítulo de sección
        doc.add_heading(seccion['subtitulo'], level=2)

        # Contenido — cada línea es un párrafo
        for linea in seccion['contenido'].strip().split('\n'):
            linea = linea.strip()
            if linea:
                # Líneas que empiezan con número son listas
                if linea[0].isdigit() or linea.startswith('-'):
                    p = doc.add_paragraph(linea,
                                          style='List Bullet')
                else:
                    doc.add_paragraph(linea)

    # Pie de página
    doc.add_paragraph('')
    pie = doc.add_paragraph(
        f'Documento: {nombre_archivo} | '
        f'Versión: 1.0 | Mayo 2026'
    )
    pie.alignment = WD_ALIGN_PARAGRAPH.CENTER

    # Guardar
    ruta = os.path.join(DOCS_PATH, nombre_archivo)
    doc.save(ruta)
    return ruta

# ── Documento 1: Estrategia Preventiva ───────────────────────
ruta = crear_docx(
    titulo = "Estrategia de Cobranza Preventiva (0-7 días)",
    secciones = [
        {
            "subtitulo": "Objetivo",
            "contenido": """
Prevenir el primer atraso o regularizar dentro de los primeros
7 días antes de que escale a mora formal.
La intervención preventiva es la más económica y efectiva.
            """
        },
        {
            "subtitulo": "Perfil del Cliente Objetivo",
            "contenido": """
- Cliente sin atraso actual pero con señales de riesgo en EWS
- Cliente con 1-7 días de atraso en la cuota actual
- Clientes con probabilidad EWS mayor al 20%
            """
        },
        {
            "subtitulo": "Acciones Recomendadas",
            "contenido": """
1. Llamada de cortesía preventiva 3 días ANTES del vencimiento
   Recordatorio amable de fecha de pago.
   Verificar que el cliente tenga los fondos disponibles.
   Tono: amigable, no intimidante.

2. SMS recordatorio 1 día antes del vencimiento
   Mensaje: recordatorio de fecha de pago y canales disponibles.

3. Si el pago no se recibe en fecha de vencimiento:
   Llamada al día siguiente (día 1 de atraso).
   Verificar si fue un olvido o dificultad real.
   Si fue olvido: dar 24-48 horas para regularizar.
   Si hay dificultad: escalar a protocolo de prórroga.
            """
        },
        {
            "subtitulo": "Oferta de Prórroga Disponible",
            "contenido": """
- Extensión de 5 días sin cargo adicional (1 vez por año)
- Requiere solicitud verbal del cliente en la llamada
- Aplicable solo a clientes con historial limpio (menos de 2 atrasos previos)
            """
        },
        {
            "subtitulo": "KPI de Éxito",
            "contenido": """
- Regularización antes del día 7 en más del 85% de los casos
- Tiempo promedio de gestión: menos de 5 minutos por llamada
            """
        },
    ],
    nombre_archivo = "01_estrategia_preventiva.docx"
)
print(f"✅ {os.path.basename(ruta)}")

# ── Documento 2: Estrategia Temprana ─────────────────────────
ruta = crear_docx(
    titulo = "Estrategia de Cobranza Temprana (8-30 días)",
    secciones = [
        {
            "subtitulo": "Objetivo",
            "contenido": """
Recuperar el pago vencido y mantener la relación con el cliente
antes de que la mora se vuelva crónica.
            """
        },
        {
            "subtitulo": "Acciones Días 8-15",
            "contenido": """
1. Llamada de gestión directa — identificar causa del no pago:
   - Problema temporal de liquidez: ofrecer plan de pago
   - Desacuerdo con el monto: escalar a revisión de cuenta
   - Pérdida de empleo: protocolo de reestructura
   - No contesta: intentar 3 veces en horarios distintos

2. Contacto con referencia personal si no hay respuesta
   Solo para confirmar que el cliente está localizable.
   NO revelar información del crédito. Cumplimiento CONDUSEF Art. 17.
            """
        },
        {
            "subtitulo": "Acciones Días 16-30",
            "contenido": """
3. Carta formal de notificación de adeudo
   Enviada por correo electrónico y mensajería certificada.
   Detalle del monto vencido más intereses moratorios.
   Fecha límite de 5 días hábiles para regularizar.

4. Oferta de quita de intereses moratorios
   Si el cliente paga el capital vencido en 48 horas:
   condonación del 100% de intereses moratorios generados.
   Válido solo en este período de 8 a 30 días.
            """
        },
        {
            "subtitulo": "KPI de Éxito",
            "contenido": """
- Recuperación en más del 70% de casos en este período
- Escalamiento a cobranza media si no hay pago al día 30
            """
        },
    ],
    nombre_archivo = "02_estrategia_temprana.docx"
)
print(f"✅ {os.path.basename(ruta)}")

# ── Documento 3: Estrategia Media ────────────────────────────
ruta = crear_docx(
    titulo = "Estrategia de Cobranza Media (31-90 días)",
    secciones = [
        {
            "subtitulo": "Objetivo",
            "contenido": """
Recuperar la cartera vencida mediante acuerdos de reestructura
antes de llegar a cartera irrecuperable.
Clasificación CNBV: Cartera vencida C1-C2. Provisión 25-75%.
            """
        },
        {
            "subtitulo": "Acciones Días 31-60",
            "contenido": """
1. Gestión intensiva: mínimo 5 intentos de contacto por semana
   Llamadas en horarios variados: mañana, tarde y noche.
   WhatsApp si el cliente tiene registrado este canal.
   Visita domiciliaria si no hay respuesta telefónica.

2. Propuesta de reestructura formal:
   Opción A: Pago al corriente con condonación del 50% de intereses moratorios.
   Opción B: Plan de pagos con 30% de enganche y resto en 3-6 cuotas.
   Opción C: Refinanciamiento absorbiendo deuda actual (requiere evaluación).
            """
        },
        {
            "subtitulo": "Acciones Días 61-90",
            "contenido": """
3. Última llamada de conciliación antes de proceso legal
   Presentar consecuencias concretas del proceso legal.
   Ofrecer la mejor condición disponible (quita máxima).
   Documentar la llamada para el expediente.
            """
        },
        {
            "subtitulo": "KPI de Éxito",
            "contenido": """
- Acuerdo de pago en más del 45% de los casos
- Toda gestión debe quedar documentada en el sistema
            """
        },
    ],
    nombre_archivo = "03_estrategia_media.docx"
)
print(f"✅ {os.path.basename(ruta)}")

# ── Documento 4: Estrategia Tardía ───────────────────────────
ruta = crear_docx(
    titulo = "Estrategia de Cobranza Tardía (91+ días)",
    secciones = [
        {
            "subtitulo": "Objetivo",
            "contenido": """
Maximizar la recuperación antes del castigo contable.
La recuperación estadística en este segmento es menor al 30%.
Clasificación CNBV: Cartera vencida C3. Provisión 100%.
            """
        },
        {
            "subtitulo": "Opciones Disponibles",
            "contenido": """
1. Oferta de quita de capital (descuento en el principal)
   Descuento del 20-40% del saldo total.
   Pago único en menos de 15 días hábiles.
   Requiere autorización del Comité de Cartera.

2. Dación en pago
   El cliente entrega un bien a valor de mercado.
   Requiere avalúo y aprobación jurídica.

3. Cesión de deuda a despacho externo
   Para cuentas sin respuesta después de 90 días de gestión.

4. Proceso legal
   Última opción después de agotar las anteriores.
   Solo para montos que justifiquen el costo legal.
            """
        },
        {
            "subtitulo": "KPI de Éxito",
            "contenido": """
- Recuperación mayor al 25% del saldo es considerada exitosa
- Documentar exhaustivamente para expediente legal si aplica
            """
        },
    ],
    nombre_archivo = "04_estrategia_tardia.docx"
)
print(f"✅ {os.path.basename(ruta)}")

# ── Documento 5: Marco Legal CONDUSEF ────────────────────────
ruta = crear_docx(
    titulo = "Marco Legal CONDUSEF — Prácticas de Cobranza",
    secciones = [
        {
            "subtitulo": "Prácticas Permitidas",
            "contenido": """
- Contactar al cliente entre las 7am y las 10pm
- Llamar hasta 3 veces al día al teléfono registrado
- Enviar notificaciones por escrito al domicilio registrado
- Contactar referencias únicamente para localizar al cliente
- Informar sobre el monto adeudado y consecuencias del impago
- Ofrecer alternativas de pago y reestructura
            """
        },
        {
            "subtitulo": "Prácticas Prohibidas",
            "contenido": """
- Llamar antes de las 7am o después de las 10pm
- Amenazar con cárcel, embargo ilegal o consecuencias falsas
- Revelar información del crédito a terceros sin autorización
- Llamar más de 3 veces al día al mismo número
- Usar lenguaje intimidante, ofensivo o denigrante
- Contactar el lugar de trabajo del cliente para presionar
- Publicar datos del deudor en redes sociales o medios
- Cobrar honorarios o gastos de cobranza no pactados
            """
        },
        {
            "subtitulo": "Derechos del Cliente",
            "contenido": """
- Recibir estado de cuenta detallado con desglose de la deuda
- Solicitar aclaración de cargos en disputa
- Recibir oferta de reestructura por escrito
- Presentar queja ante CONDUSEF sin represalias
- Solicitar que el despacho externo deje de contactarlo
            """
        },
        {
            "subtitulo": "Contacto CONDUSEF",
            "contenido": """
- Teléfono CONDUSEF: 800-999-8080
- Portal de quejas: www.condusef.gob.mx
- Toda gestión de cobranza debe respetar estas disposiciones
            """
        },
    ],
    nombre_archivo = "05_marco_legal_condusef.docx"
)
print(f"✅ {os.path.basename(ruta)}")

# ── Documento 6: Perfiles de Cliente ─────────────────────────
ruta = crear_docx(
    titulo = "Perfiles de Cliente y Estrategia Diferenciada",
    secciones = [
        {
            "subtitulo": "Perfil A — Cliente Recurrente con Buen Historial",
            "contenido": """
Características:
- 2 o más créditos previos pagados
- Historial de atrasos menor a 15 días en promedio
- Relación mayor a 2 años con la institución

Estrategia:
- Trato preferencial y empático
- Ofrecer prórroga automática sin trámite
- Llamada de un solo intento antes de escalar
- No contactar referencias en primera instancia
            """
        },
        {
            "subtitulo": "Perfil B — Cliente Nuevo o con Poco Historial",
            "contenido": """
Características:
- Primer crédito o menos de 6 meses de relación
- Sin historial previo de atrasos

Estrategia:
- Educación financiera en la llamada
- Explicar consecuencias del atraso de forma didáctica
- Ofrecer orientación sobre canales de pago
- Enfoque en la relación a largo plazo
            """
        },
        {
            "subtitulo": "Perfil C — Cliente con Historial de Atrasos Recurrentes",
            "contenido": """
Características:
- Patrón de atrasos en 30% o más de sus cuotas históricas
- Múltiples gestiones de cobranza previas

Estrategia:
- Gestión más directa y formal desde el primer contacto
- No ofrecer extensiones adicionales
- Documentar exhaustivamente cada intento de contacto
- Escalar más rápidamente a cobranza media si no hay pago
            """
        },
        {
            "subtitulo": "Perfil D — Cliente en Situación Vulnerable",
            "contenido": """
Características:
- Pérdida de empleo documentada
- Enfermedad grave propia o familiar
- Siniestro como robo, incendio o desastre natural

Estrategia:
- Protocolo de empatía: escuchar antes de proponer
- Ofrecer período de gracia inmediato de 1 a 3 meses
- Conectar con programa de apoyo de la institución
- Nunca presionar en la primera llamada
- Seguimiento proactivo a los 30 días
            """
        },
    ],
    nombre_archivo = "06_perfiles_cliente.docx"
)
print(f"✅ {os.path.basename(ruta)}")

# ── Documento 7: Clasificación CNBV (.xlsx) ───────────────────
# Este documento es tabular — naturalmente vive en Excel
datos_cnbv = {
    'Días de Atraso': [
        '0 días', '1-30 días', '31-60 días',
        '61-90 días', '91-180 días', '180+ días'
    ],
    'Clasificación CNBV': [
        'Cartera Vigente', 'Cartera en Riesgo C1',
        'Cartera Vencida C2A', 'Cartera Vencida C2B',
        'Cartera Vencida C3', 'Cartera Irrecuperable'
    ],
    'Provisión Requerida': [
        '0.5% - 2%', '10% - 25%', '25% - 50%',
        '50% - 75%', '75% - 100%', '100%'
    ],
    'Acción de Cobranza': [
        'Sin acción / Preventiva EWS',
        'Cobranza temprana',
        'Cobranza media — reestructura',
        'Cobranza media intensiva',
        'Cobranza tardía — quita',
        'Castigo contable / proceso legal'
    ],
    'Estrategia Documento': [
        '01_estrategia_preventiva.docx',
        '02_estrategia_temprana.docx',
        '03_estrategia_media.docx',
        '03_estrategia_media.docx',
        '04_estrategia_tardia.docx',
        '04_estrategia_tardia.docx'
    ]
}

df_cnbv = pd.DataFrame(datos_cnbv)
ruta_xlsx = os.path.join(DOCS_PATH, '07_clasificacion_cnbv.xlsx')
df_cnbv.to_excel(ruta_xlsx, index=False)
print(f"✅ 07_clasificacion_cnbv.xlsx")

# ── Resumen final ─────────────────────────────────────────────
print(f"\n📁 Documentos creados en: {DOCS_PATH}")
print(f"\n{'Archivo':<45} {'Tamaño':>10}")
print("-" * 57)

total_mb = 0
for archivo in sorted(os.listdir(DOCS_PATH)):
    ruta_a  = os.path.join(DOCS_PATH, archivo)
    tam_kb  = os.path.getsize(ruta_a) / 1024
    total_mb += tam_kb / 1024
    print(f"   {archivo:<43} {tam_kb:>8.1f} KB")

print(f"\n   Total: {total_mb:.2f} MB")
print(f"\n✅ Base de conocimiento lista para indexar en ChromaDB")
print(f"   Para actualizar: reemplaza los archivos por los")
print(f"   documentos oficiales de la institución.")

✅ 01_estrategia_preventiva.docx
✅ 02_estrategia_temprana.docx
✅ 03_estrategia_media.docx
✅ 04_estrategia_tardia.docx
✅ 05_marco_legal_condusef.docx
✅ 06_perfiles_cliente.docx
✅ 07_clasificacion_cnbv.xlsx

📁 Documentos creados en: C:\Users\Marin\Documents\PROYECTO ML_OPS\credit-risk-scoring-ml\src\collections\documentos

Archivo                                           Tamaño
---------------------------------------------------------
   01_estrategia_preventiva.docx                   36.6 KB
   02_estrategia_temprana.docx                     36.5 KB
   03_estrategia_media.docx                        36.5 KB
   04_estrategia_tardia.docx                       36.4 KB
   05_marco_legal_condusef.docx                    36.6 KB
   06_perfiles_cliente.docx                        36.7 KB
   07_clasificacion_cnbv.xlsx                       5.2 KB

   Total: 0.22 MB

✅ Base de conocimiento lista para indexar en ChromaDB
   Para actualizar: reemplaza los archivos por los
   documentos oficiales d

In [4]:
# ============================================================
# CELDA 4 — CARGA DINÁMICA DE DOCUMENTOS
# ============================================================
# Objetivo: leer todos los archivos de la carpeta documentos/
# sin depender de nombres específicos, extraer su texto
# y dividirlos en chunks para indexar en ChromaDB.
#
# Carga dinámica: el sistema lee lo que haya en la carpeta.
# Para actualizar documentos: solo reemplazar los archivos.
# El código no necesita modificarse.
#
# Formatos soportados: .docx, .xlsx, .txt, .pdf
# ============================================================

from docx import Document as DocxDocument
import fitz          # pymupdf — para PDFs
import openpyxl

# ── Configuración del chunking ───────────────────────────────
# CHUNK_SIZE: tamaño máximo de cada fragmento en caracteres
# Muy grande → menos precisión en búsqueda
# Muy pequeño → pierde contexto, demasiados fragmentos
CHUNK_SIZE    = 800

# CHUNK_OVERLAP: cuántos caracteres se repiten entre chunks
# Evita que una idea importante quede cortada entre dos chunks
CHUNK_OVERLAP = 100

# ── Funciones de extracción de texto por formato ─────────────

def extraer_texto_docx(ruta):
    """Extrae texto de un archivo Word (.docx)."""
    doc   = DocxDocument(ruta)
    texto = []
    for parrafo in doc.paragraphs:
        if parrafo.text.strip():
            texto.append(parrafo.text.strip())
    return '\n'.join(texto)

def extraer_texto_pdf(ruta):
    """Extrae texto de un archivo PDF usando pymupdf."""
    doc   = fitz.open(ruta)
    texto = []
    for pagina in doc:
        texto.append(pagina.get_text())
    doc.close()
    return '\n'.join(texto)

def extraer_texto_xlsx(ruta):
    """
    Extrae texto de un archivo Excel.
    Convierte cada fila en una oración descriptiva
    para que sea buscable semánticamente.
    """
    wb    = openpyxl.load_workbook(ruta)
    texto = []

    for hoja in wb.sheetnames:
        ws      = wb[hoja]
        headers = []

        for i, fila in enumerate(ws.iter_rows(values_only=True)):
            # Primera fila son los encabezados
            if i == 0:
                headers = [str(h) for h in fila if h]
                continue

            # Convertir cada fila en una oración descriptiva
            if any(celda for celda in fila):
                partes = []
                for j, valor in enumerate(fila):
                    if valor and j < len(headers):
                        partes.append(
                            f"{headers[j]}: {valor}"
                        )
                if partes:
                    texto.append(' | '.join(partes))

    return '\n'.join(texto)

def extraer_texto_txt(ruta):
    """Extrae texto de un archivo de texto plano."""
    with open(ruta, 'r', encoding='utf-8',
              errors='ignore') as f:
        return f.read()

def dividir_en_chunks(texto, chunk_size=CHUNK_SIZE,
                       overlap=CHUNK_OVERLAP):
    """
    Divide un texto en fragmentos de tamaño chunk_size
    con overlap entre fragmentos consecutivos.

    El overlap garantiza que ideas que cruzan el límite
    de un chunk aparezcan en ambos fragmentos.
    """
    chunks = []
    inicio = 0

    while inicio < len(texto):
        fin = inicio + chunk_size

        # Si no es el último chunk, buscar el último punto
        # o salto de línea para no cortar a mitad de oración
        if fin < len(texto):
            # Buscar el último punto o salto de línea
            # dentro del rango del chunk
            ultimo_punto = max(
                texto.rfind('.', inicio, fin),
                texto.rfind('\n', inicio, fin)
            )
            if ultimo_punto > inicio:
                fin = ultimo_punto + 1

        chunk = texto[inicio:fin].strip()
        if chunk:
            chunks.append(chunk)

        # El siguiente chunk empieza con overlap
        inicio = fin - overlap

    return chunks

# ── Carga dinámica — lee todos los archivos de la carpeta ────
EXTENSIONES_SOPORTADAS = {'.docx', '.pdf', '.xlsx', '.txt'}

print("=" * 55)
print("CARGA DINÁMICA DE DOCUMENTOS")
print("=" * 55)
print(f"\nCarpeta de documentos: {DOCS_PATH}")
print(f"Chunk size  : {CHUNK_SIZE} caracteres")
print(f"Chunk overlap: {CHUNK_OVERLAP} caracteres")
print()

todos_los_chunks = []  # Lista de todos los fragmentos
errores          = []  # Archivos que fallaron

archivos = sorted([
    f for f in os.listdir(DOCS_PATH)
    if os.path.splitext(f)[1].lower() in EXTENSIONES_SOPORTADAS
])

print(f"Archivos encontrados: {len(archivos)}")
print("-" * 55)

for archivo in archivos:
    ruta_archivo = os.path.join(DOCS_PATH, archivo)
    extension    = os.path.splitext(archivo)[1].lower()

    try:
        # Extraer texto según el formato
        if extension == '.docx':
            texto = extraer_texto_docx(ruta_archivo)
        elif extension == '.pdf':
            texto = extraer_texto_pdf(ruta_archivo)
        elif extension == '.xlsx':
            texto = extraer_texto_xlsx(ruta_archivo)
        elif extension == '.txt':
            texto = extraer_texto_txt(ruta_archivo)

        # Dividir en chunks
        chunks = dividir_en_chunks(texto)

        # Guardar cada chunk con metadata del archivo origen
        for i, chunk in enumerate(chunks):
            todos_los_chunks.append({
                'id'      : f"{archivo}__chunk_{i:03d}",
                'texto'   : chunk,
                'archivo' : archivo,
                'chunk_n' : i,
                'formato' : extension[1:],  # sin el punto
            })

        print(f"  ✅ {archivo:<45} "
              f"{len(chunks):>3} chunks  "
              f"({len(texto):,} chars)")

    except Exception as e:
        errores.append({'archivo': archivo, 'error': str(e)})
        print(f"  ❌ {archivo:<45} ERROR: {e}")

# ── Resumen ───────────────────────────────────────────────────
print(f"\n{'='*55}")
print(f"RESUMEN DE CARGA")
print(f"{'='*55}")
print(f"  Archivos procesados : {len(archivos) - len(errores)}")
print(f"  Archivos con error  : {len(errores)}")
print(f"  Total de chunks     : {len(todos_los_chunks)}")
print(f"  Promedio chunks/doc : "
      f"{len(todos_los_chunks)/max(len(archivos),1):.1f}")

if errores:
    print(f"\n  ⚠️  Errores:")
    for e in errores:
        print(f"     {e['archivo']}: {e['error']}")

print(f"\n✅ Documentos listos para indexar en ChromaDB")

CARGA DINÁMICA DE DOCUMENTOS

Carpeta de documentos: C:\Users\Marin\Documents\PROYECTO ML_OPS\credit-risk-scoring-ml\src\collections\documentos
Chunk size  : 800 caracteres
Chunk overlap: 100 caracteres

Archivos encontrados: 7
-------------------------------------------------------
  ✅ 01_estrategia_preventiva.docx                   3 chunks  (1,451 chars)
  ✅ 02_estrategia_temprana.docx                     2 chunks  (1,291 chars)
  ✅ 03_estrategia_media.docx                        2 chunks  (1,174 chars)
  ✅ 04_estrategia_tardia.docx                       2 chunks  (997 chars)
  ✅ 05_marco_legal_condusef.docx                    3 chunks  (1,409 chars)
  ✅ 06_perfiles_cliente.docx                        3 chunks  (1,693 chars)
  ✅ 07_clasificacion_cnbv.xlsx                      2 chunks  (1,194 chars)

RESUMEN DE CARGA
  Archivos procesados : 7
  Archivos con error  : 0
  Total de chunks     : 17
  Promedio chunks/doc : 2.4

✅ Documentos listos para indexar en ChromaDB


In [6]:
# ============================================================
# CELDA 5 — CONSTRUIR VECTOR STORE CON CHROMADB
# ============================================================
# Objetivo: convertir los chunks de texto en vectores
# numéricos y guardarlos en ChromaDB para búsqueda semántica.
#
# Proceso:
# 1. Cargar el modelo de embeddings (sentence-transformers)
# 2. Convertir cada chunk en un vector de 384 dimensiones
# 3. Guardar vectores + texto + metadata en ChromaDB
#
# ¿Por qué 384 dimensiones?
# El modelo MiniLM convierte cualquier texto en un vector
# de 384 números que captura su significado semántico.
# Textos con significado similar tienen vectores similares
# (distancia coseno pequeña).
#
# Persistencia:
# ChromaDB guarda los vectores en disco (CHROMA_PATH).
# Si reinicias el kernel, no necesitas re-indexar —
# solo cargar el cliente de ChromaDB desde la misma ruta.
# ============================================================


# ── Fix SSL para entorno conda en Windows ────────────────────
# La variable SSL_CERT_FILE apunta a un archivo inexistente
# en este entorno conda. Se limpia para que Python use
# los certificados SSL del sistema operativo.
import os
if 'SSL_CERT_FILE' in os.environ:
    del os.environ['SSL_CERT_FILE']
if 'REQUESTS_CA_BUNDLE' in os.environ:
    del os.environ['REQUESTS_CA_BUNDLE']



import chromadb
from sentence_transformers import SentenceTransformer
import time

print("=" * 55)
print("CONSTRUYENDO VECTOR STORE — CHROMADB")
print("=" * 55)

# ── Paso 1: Cargar modelo de embeddings ──────────────────────
# La primera vez descarga el modelo (~118MB)
# Las siguientes veces lo carga desde caché local
print("\n[1/4] Cargando modelo de embeddings...")
print("      paraphrase-multilingual-MiniLM-L12-v2")
print("      (primera vez descarga ~118MB, después es instantáneo)")

inicio = time.time()
modelo_embeddings = SentenceTransformer(
    'paraphrase-multilingual-MiniLM-L12-v2'
)
tiempo = time.time() - inicio
print(f"      ✅ Modelo cargado en {tiempo:.1f} segundos")
print(f"      Dimensiones del vector: "
      f"{modelo_embeddings.get_sentence_embedding_dimension()}")

# ── Paso 2: Inicializar ChromaDB persistente ─────────────────
# PersistentClient guarda los vectores en disco
# Si el kernel se reinicia, los vectores siguen ahí
print(f"\n[2/4] Inicializando ChromaDB...")

chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)

# Crear o recuperar la colección de documentos de cobranza
# Si ya existe (re-ejecución), la elimina y recrea
# para garantizar que los documentos estén actualizados
COLLECTION_NAME = "cobranza_knowledge_base"

colecciones_existentes = [
    c.name for c in chroma_client.list_collections()
]

if COLLECTION_NAME in colecciones_existentes:
    chroma_client.delete_collection(COLLECTION_NAME)
    print(f"      ♻️  Colección anterior eliminada — re-indexando")

coleccion = chroma_client.create_collection(
    name     = COLLECTION_NAME,
    metadata = {
        "descripcion": "Base de conocimiento cobranza",
        "version"    : "1.0",
        "fecha"      : "2026-05",
        "n_docs"     : str(len(archivos))
    }
)
print(f"      ✅ Colección '{COLLECTION_NAME}' creada")

# ── Paso 3: Generar embeddings e indexar ─────────────────────
# Para cada chunk:
# 1. Generar su vector con sentence-transformers
# 2. Guardarlo en ChromaDB con su texto y metadata
print(f"\n[3/4] Generando embeddings e indexando {len(todos_los_chunks)} chunks...")

# Procesamos en lotes para eficiencia
BATCH_SIZE = 10   # Chunks por lote

for i in range(0, len(todos_los_chunks), BATCH_SIZE):
    lote = todos_los_chunks[i:i + BATCH_SIZE]

    # Extraer textos, ids y metadata del lote
    textos    = [chunk['texto']   for chunk in lote]
    ids       = [chunk['id']      for chunk in lote]
    metadatas = [{
        'archivo' : chunk['archivo'],
        'chunk_n' : str(chunk['chunk_n']),
        'formato' : chunk['formato'],
    } for chunk in lote]

    # Generar embeddings para el lote completo
    # show_progress_bar=False para output limpio
    embeddings = modelo_embeddings.encode(
        textos,
        show_progress_bar = False
    ).tolist()

    # Guardar en ChromaDB
    coleccion.add(
        ids        = ids,
        documents  = textos,
        embeddings = embeddings,
        metadatas  = metadatas
    )

    chunks_procesados = min(i + BATCH_SIZE,
                             len(todos_los_chunks))
    print(f"      Indexados: {chunks_procesados:>3}/"
          f"{len(todos_los_chunks)} chunks")

print(f"\n      ✅ Todos los chunks indexados")

# ── Paso 4: Verificar la indexación ──────────────────────────
print(f"\n[4/4] Verificando indexación...")

n_docs = coleccion.count()
print(f"      Chunks en ChromaDB : {n_docs}")

# Prueba de búsqueda semántica
print(f"\n      🔍 Prueba de búsqueda semántica:")
query_prueba = "cliente con 45 días de atraso estrategia de cobranza"

embedding_query = modelo_embeddings.encode(
    [query_prueba]
).tolist()

resultados = coleccion.query(
    query_embeddings = embedding_query,
    n_results        = 3
)

print(f"      Query: '{query_prueba}'")
print(f"\n      Top 3 resultados:")
for j, (doc, meta, dist) in enumerate(zip(
    resultados['documents'][0],
    resultados['metadatas'][0],
    resultados['distances'][0]
), 1):
    print(f"\n      Resultado {j}:")
    print(f"      Archivo  : {meta['archivo']}")
    print(f"      Distancia: {dist:.4f} "
          f"(menor = más similar)")
    print(f"      Texto    : {doc[:150]}...")

# ── Guardar referencia del cliente para uso posterior ────────
# Guardamos el path de ChromaDB para que el dashboard
# pueda cargarlo sin re-indexar
chroma_config = {
    'chroma_path'     : CHROMA_PATH,
    'collection_name' : COLLECTION_NAME,
    'modelo_embeddings': 'paraphrase-multilingual-MiniLM-L12-v2',
    'chunk_size'      : CHUNK_SIZE,
    'chunk_overlap'   : CHUNK_OVERLAP,
    'n_chunks'        : len(todos_los_chunks),
    'n_documentos'    : len(archivos),
}

import joblib
joblib.dump(
    chroma_config,
    os.path.join(MODELS_PATH, 'chroma_config.pkl')
)

print(f"\n{'='*55}")
print(f"VECTOR STORE CONSTRUIDO EXITOSAMENTE")
print(f"{'='*55}")
print(f"  Chunks indexados    : {n_docs}")
print(f"  Modelo embeddings   : "
      f"paraphrase-multilingual-MiniLM-L12-v2")
print(f"  Dimensiones vector  : "
      f"{modelo_embeddings.get_sentence_embedding_dimension()}")
print(f"  Persistencia en     : {CHROMA_PATH}")
print(f"  Config guardada en  : {MODELS_PATH}/chroma_config.pkl")
print(f"\n✅ ChromaDB listo para consultas RAG")

CONSTRUYENDO VECTOR STORE — CHROMADB

[1/4] Cargando modelo de embeddings...
      paraphrase-multilingual-MiniLM-L12-v2
      (primera vez descarga ~118MB, después es instantáneo)


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

      ✅ Modelo cargado en 434.8 segundos
      Dimensiones del vector: 384

[2/4] Inicializando ChromaDB...
      ✅ Colección 'cobranza_knowledge_base' creada

[3/4] Generando embeddings e indexando 17 chunks...
      Indexados:  10/17 chunks
      Indexados:  17/17 chunks

      ✅ Todos los chunks indexados

[4/4] Verificando indexación...
      Chunks en ChromaDB : 17

      🔍 Prueba de búsqueda semántica:
      Query: 'cliente con 45 días de atraso estrategia de cobranza'

      Top 3 resultados:

      Resultado 1:
      Archivo  : 01_estrategia_preventiva.docx
      Distancia: 6.3904 (menor = más similar)
      Texto    : datorio 1 día antes del vencimiento
Mensaje: recordatorio de fecha de pago y canales disponibles.
3. Si el pago no se recibe en fecha de vencimiento:
...

      Resultado 2:
      Archivo  : 06_perfiles_cliente.docx
      Distancia: 7.8873 (menor = más similar)
      Texto    : - Explicar consecuencias del atraso de forma didáctica
- Ofrecer orientación sobre can

In [8]:
# ============================================================
# CELDA 6 — FUNCIÓN DE CONSULTA RAG (versión 2)
# ============================================================
# Objetivo: dado un cliente en riesgo, recuperar los
# documentos más relevantes de ChromaDB y estructurarlos
# para el prompt de Claude API.
#
# Versión 2 mejoras:
# - Búsqueda en dos pasos: documento prioritario por nivel
#   de mora + documentos complementarios semánticos
# - Garantiza que el documento correcto por rango de días
#   siempre aparezca primero, independientemente de la
#   similitud semántica
#
# Flujo:
# 1. Construir query semántica desde el perfil del cliente
# 2. Paso 1: recuperar documento prioritario por nivel mora
# 3. Paso 2: recuperar documentos complementarios
# 4. Combinar y formatear contexto para Claude API
# ============================================================

def construir_query_semantica(cliente_info: dict) -> str:
    """
    Construye una query de búsqueda semántica basada
    en el perfil del cliente.

    Mientras más descriptiva sea la query, más relevantes
    serán los documentos recuperados por similitud semántica.

    Parámetros:
    -----------
    cliente_info : dict con información del cliente

    Retorna:
    --------
    str — query descriptiva para búsqueda semántica
    """
    # ── Determinar nivel de mora ─────────────────────────────
    # Esta descripción textual del nivel de mora hace que
    # el embedding de la query sea similar al de los
    # documentos de estrategia correspondientes
    dias = cliente_info.get('max_dias_atraso', 0)

    if dias == 0:
        nivel = "preventiva sin atraso actual"
    elif dias <= 7:
        nivel = "preventiva 1 a 7 días de atraso"
    elif dias <= 30:
        nivel = "temprana 8 a 30 días de atraso"
    elif dias <= 90:
        nivel = "media 31 a 90 días de atraso reestructura"
    else:
        nivel = "tardía más de 90 días quita castigo"

    # ── Determinar perfil del cliente ────────────────────────
    es_recurrente = cliente_info.get(
        'es_cliente_recurrente', 0
    )
    carga = cliente_info.get('carga_financiera', 0)

    if es_recurrente:
        perfil = "cliente recurrente con historial previo"
    else:
        perfil = "cliente nuevo sin historial previo"

    if carga > 0.5:
        perfil += " carga financiera alta"

    # ── Construir query final ────────────────────────────────
    query = (
        f"estrategia cobranza {nivel} "
        f"{perfil} "
        f"acciones recuperacion protocolo contacto"
    )

    return query


def determinar_archivo_prioritario(dias_atraso: int) -> str:
    """
    Determina qué documento de estrategia corresponde
    al nivel de mora del cliente según sus días de atraso.

    Esta función implementa la lógica de negocio de cobranza:
    cada rango de días tiene una estrategia específica.

    Parámetros:
    -----------
    dias_atraso : int — días de atraso del cliente

    Retorna:
    --------
    str — nombre del archivo de estrategia correspondiente
    """
    if dias_atraso <= 7:
        return "01_estrategia_preventiva.docx"
    elif dias_atraso <= 30:
        return "02_estrategia_temprana.docx"
    elif dias_atraso <= 90:
        return "03_estrategia_media.docx"
    else:
        return "04_estrategia_tardia.docx"


def consultar_rag(cliente_info: dict,
                  n_resultados: int = 4) -> dict:
    """
    Consulta el vector store y recupera documentos
    relevantes para el perfil del cliente.

    Búsqueda en dos pasos:
    1. Documento prioritario: el que corresponde exactamente
       al nivel de mora del cliente (por lógica de negocio)
    2. Documentos complementarios: los más similares
       semánticamente para dar contexto adicional (perfiles,
       marco legal, protocolos)

    Parámetros:
    -----------
    cliente_info : dict con información del cliente
    n_resultados : número de documentos complementarios

    Retorna:
    --------
    dict con query, documentos y contexto_rag formateado
    """
    # ── Construir query semántica ────────────────────────────
    query = construir_query_semantica(cliente_info)

    # ── Generar embedding de la query ────────────────────────
    embedding_query = modelo_embeddings.encode(
        [query]
    ).tolist()

    # ── Determinar documento prioritario ────────────────────
    # El documento prioritario se elige por lógica de negocio
    # (rango de días de atraso), NO por similitud semántica.
    # Esto garantiza que el documento correcto siempre
    # aparezca primero, independientemente del embedding.
    dias = cliente_info.get('max_dias_atraso', 0)
    archivo_prioritario = determinar_archivo_prioritario(dias)

    # ── Paso 1: Recuperar documento prioritario ──────────────
    # Filtra ChromaDB por el archivo exacto que corresponde
    # al nivel de mora del cliente
    resultado_prioritario = coleccion.query(
        query_embeddings = embedding_query,
        n_results        = 1,
        where            = {"archivo": archivo_prioritario}
    )

    # ── Paso 2: Recuperar documentos complementarios ─────────
    # Busca semánticamente excluyendo el ya recuperado
    # para no duplicar información
    resultado_complementario = coleccion.query(
        query_embeddings = embedding_query,
        n_results        = n_resultados,
        where            = {
            "archivo": {"$ne": archivo_prioritario}
        }
    )

    # ── Combinar resultados ──────────────────────────────────
    # El prioritario va primero — es el más importante
    documentos = []

    if resultado_prioritario['documents'][0]:
        documentos.append({
            'archivo'   : resultado_prioritario[
                'metadatas'][0][0]['archivo'],
            'contenido' : resultado_prioritario[
                'documents'][0][0],
            'distancia' : resultado_prioritario[
                'distances'][0][0],
            'tipo'      : 'prioritario'
        })

    for doc, meta, dist in zip(
        resultado_complementario['documents'][0],
        resultado_complementario['metadatas'][0],
        resultado_complementario['distances'][0]
    ):
        documentos.append({
            'archivo'   : meta['archivo'],
            'contenido' : doc,
            'distancia' : dist,
            'tipo'      : 'complementario'
        })

    # ── Formatear contexto para Claude ───────────────────────
    # Cada documento va claramente etiquetado para que
    # Claude pueda distinguir el documento principal
    # de los complementarios
    contexto_partes = []
    for i, doc in enumerate(documentos, 1):
        nombre = (
            doc['archivo']
            .replace('.docx', '')
            .replace('.xlsx', '')
            .replace('_', ' ')
            .title()
        )
        tipo = (
            "📌 PRIORITARIO"
            if doc['tipo'] == 'prioritario'
            else "📄 Complementario"
        )
        contexto_partes.append(
            f"[{tipo} — {nombre}]\n{doc['contenido']}"
        )

    contexto_rag = "\n\n".join(contexto_partes)

    return {
        'query'       : query,
        'documentos'  : documentos,
        'contexto_rag': contexto_rag
    }


# ── Prueba de la función RAG mejorada ────────────────────────
print("=" * 55)
print("PRUEBA DE FUNCIÓN RAG — VERSIÓN 2")
print("=" * 55)

clientes_prueba = [
    {
        'nombre'              : 'Cliente A — Mora media',
        'max_dias_atraso'     : 45,
        'probabilidad_ews'    : 0.78,
        'es_cliente_recurrente': 1,
        'carga_financiera'    : 0.35,
        'n_cuotas_total'      : 18,
    },
    {
        'nombre'              : 'Cliente B — Sin mora, alto riesgo EWS',
        'max_dias_atraso'     : 0,
        'probabilidad_ews'    : 0.65,
        'es_cliente_recurrente': 0,
        'carga_financiera'    : 0.72,
        'n_cuotas_total'      : 3,
    }
]

for cliente in clientes_prueba:
    print(f"\n{'─'*55}")
    print(f"Perfil: {cliente['nombre']}")
    print(f"  Días atraso     : {cliente['max_dias_atraso']}")
    print(f"  Prob. EWS       : {cliente['probabilidad_ews']:.0%}")
    print(f"  Recurrente      : "
          f"{'Sí' if cliente['es_cliente_recurrente'] else 'No'}")
    print(f"  Carga financiera: {cliente['carga_financiera']:.0%}")

    resultado = consultar_rag(cliente, n_resultados=3)

    print(f"\n  Query generada:")
    print(f"  {resultado['query']}")
    print(f"\n  Documentos recuperados:")

    for i, doc in enumerate(resultado['documentos'], 1):
        tipo  = "📌" if doc['tipo'] == 'prioritario' else "📄"
        print(f"    {tipo} {i}. {doc['archivo']:<43} "
              f"dist={doc['distancia']:.2f}")

print(f"\n✅ Función RAG v2 lista")

PRUEBA DE FUNCIÓN RAG — VERSIÓN 2

───────────────────────────────────────────────────────
Perfil: Cliente A — Mora media
  Días atraso     : 45
  Prob. EWS       : 78%
  Recurrente      : Sí
  Carga financiera: 35%

  Query generada:
  estrategia cobranza media 31 a 90 días de atraso reestructura cliente recurrente con historial previo acciones recuperacion protocolo contacto

  Documentos recuperados:
    📌 1. 03_estrategia_media.docx                    dist=8.16
    📄 2. 06_perfiles_cliente.docx                    dist=7.04
    📄 3. 01_estrategia_preventiva.docx               dist=7.80
    📄 4. 06_perfiles_cliente.docx                    dist=8.51

───────────────────────────────────────────────────────
Perfil: Cliente B — Sin mora, alto riesgo EWS
  Días atraso     : 0
  Prob. EWS       : 65%
  Recurrente      : No
  Carga financiera: 72%

  Query generada:
  estrategia cobranza preventiva sin atraso actual cliente nuevo sin historial previo carga financiera alta acciones recuperac

In [10]:
# ============================================================
# CELDA 7 — INTEGRACIÓN CON CLAUDE API
# ============================================================
# Objetivo: generar estrategias de cobranza personalizadas
# combinando:
#   1. Perfil del cliente (datos del EWS)
#   2. Explicabilidad SHAP (por qué está en riesgo)
#   3. Documentos RAG (estrategias y protocolos)
#
# El prompt tiene tres secciones:
#   - SISTEMA: define el rol y restricciones del agente
#   - CONTEXTO RAG: documentos recuperados de ChromaDB
#   - SOLICITUD: datos del cliente + instrucción específica
#
# La API key se lee en este orden de prioridad:
#   1. Variable de entorno ANTHROPIC_API_KEY
#   2. Archivo .env en la raíz del proyecto
#   3. dashboard/.streamlit/secrets.toml
# ============================================================

import anthropic
import os

# ── Inicializar cliente de Claude API ────────────────────────
def obtener_claude_client():
    """
    Inicializa el cliente de Claude API.
    Lee la API key en este orden:
    1. Variable de entorno ANTHROPIC_API_KEY
    2. Archivo .env en la raíz del proyecto
    3. Streamlit secrets (.streamlit/secrets.toml)
    """
    api_key = os.environ.get('ANTHROPIC_API_KEY')

    if not api_key:
        # Opción 2: archivo .env en la raíz del proyecto
        env_path = os.path.join(BASE, '.env')
        if os.path.exists(env_path):
            with open(env_path, 'r') as f:
                for linea in f:
                    if linea.startswith('ANTHROPIC_API_KEY'):
                        api_key = linea.split('=')[1].strip()\
                                       .strip('"').strip("'")
                        break

    if not api_key:
        # Opción 3: Streamlit secrets.toml
        secrets_path = os.path.join(
            BASE, 'dashboard', '.streamlit', 'secrets.toml'
        )
        if os.path.exists(secrets_path):
            with open(secrets_path, 'r') as f:
                for linea in f:
                    if 'ANTHROPIC_API_KEY' in linea:
                        api_key = linea.split('=')[1].strip()\
                                       .strip('"').strip("'")
                        break

    if not api_key:
        raise ValueError(
            "ANTHROPIC_API_KEY no encontrada en:\n"
            "  1. Variable de entorno\n"
            "  2. .env en raíz del proyecto\n"
            "  3. dashboard/.streamlit/secrets.toml"
        )

    return anthropic.Anthropic(api_key=api_key)


def formatear_perfil_cliente(cliente_info: dict,
                              shap_top: list = None) -> str:
    """
    Formatea la información del cliente para el prompt.

    Parámetros:
    -----------
    cliente_info : dict con datos del cliente
    shap_top     : lista de top variables SHAP con sus valores

    Retorna:
    --------
    str — perfil formateado para incluir en el prompt
    """
    dias   = cliente_info.get('max_dias_atraso', 0)
    prob   = cliente_info.get('probabilidad_ews', 0)
    carga  = cliente_info.get('carga_financiera', 0)
    rec    = cliente_info.get('es_cliente_recurrente', 0)
    cuotas = cliente_info.get('n_cuotas_total', 0)
    dpd    = cliente_info.get('max_dpd', 0)

    # Determinar nivel de mora para el texto
    if dias == 0:
        nivel_mora = "Sin atraso actual (alerta preventiva EWS)"
    elif dias <= 7:
        nivel_mora = f"{dias} días — Alerta temprana"
    elif dias <= 30:
        nivel_mora = f"{dias} días — Mora leve"
    elif dias <= 90:
        nivel_mora = f"{dias} días — Mora media"
    else:
        nivel_mora = f"{dias} días — Mora grave"

    perfil = f"""
DATOS DEL CLIENTE EN RIESGO:
─────────────────────────────────────────
Probabilidad de mora (EWS)  : {prob:.1%}
Días de atraso actual       : {nivel_mora}
Máx. DPD mensual histórico  : {dpd:.0f} días
Carga financiera            : {carga:.1%} del ingreso mensual
Total de cuotas del crédito : {cuotas:.0f} cuotas
Cliente recurrente          : {'Sí — tiene historial previo'
                               if rec else
                               'No — primer crédito'}
    """.strip()

    # Agregar top variables SHAP si están disponibles
    if shap_top:
        perfil += "\n\nFACTORES DE RIESGO IDENTIFICADOS (SHAP):"
        perfil += "\n─────────────────────────────────────────"
        for var in shap_top:
            direccion = "↑" if var['shap'] > 0 else "↓"
            perfil += (
                f"\n{direccion} {var['nombre']:<35} "
                f"contribución: {var['shap']:+.3f}"
            )

    return perfil


def generar_estrategia_cobranza(cliente_info: dict,
                                 shap_top: list = None,
                                 modo: str = 'completo') -> dict:
    """
    Genera una estrategia de cobranza personalizada
    usando RAG + Claude API.

    Parámetros:
    -----------
    cliente_info : dict con datos del cliente del EWS
    shap_top     : top variables SHAP del cliente
    modo         : 'completo' (estrategia detallada) o
                   'resumen' (bullet points ejecutivos)

    Retorna:
    --------
    dict con estrategia generada y metadata
    """
    # ── Paso 1: Consultar RAG ────────────────────────────────
    resultado_rag = consultar_rag(cliente_info, n_resultados=3)

    # ── Paso 2: Formatear perfil del cliente ─────────────────
    perfil_cliente = formatear_perfil_cliente(
        cliente_info, shap_top
    )

    # ── Paso 3: Construir el prompt ──────────────────────────
    # SISTEMA: define el rol del agente y sus restricciones
    # CONTEXTO: documentos RAG recuperados de ChromaDB
    # SOLICITUD: datos del cliente e instrucción específica
    prompt_sistema = """Eres el Agente de Cobranza del sistema de
inteligencia crediticia. Tu rol es generar estrategias de cobranza
personalizadas y accionables para el equipo de gestores.

INSTRUCCIONES:
- Basa tu estrategia ÚNICAMENTE en los documentos de contexto
  proporcionados y los datos del cliente
- Sé específico y accionable — el gestor debe saber exactamente
  qué hacer hoy
- Respeta el marco legal CONDUSEF en todas las recomendaciones
- Adapta el tono según el perfil del cliente (recurrente vs nuevo,
  carga financiera alta vs baja)
- Si el cliente tiene carga financiera mayor al 50%, considera
  siempre opciones de reestructura
- Máximo 300 palabras en la respuesta"""

    prompt_usuario = f"""
{perfil_cliente}

DOCUMENTOS DE REFERENCIA (Base de conocimiento de cobranza):
{'='*60}
{resultado_rag['contexto_rag']}
{'='*60}

TAREA:
Genera una estrategia de cobranza específica para este cliente.
Incluye:
1. Acción inmediata (qué hacer HOY)
2. Guión sugerido para la llamada (2-3 líneas clave)
3. Oferta a presentar (si aplica)
4. Escalamiento (si no hay respuesta en 48 horas)
5. Restricciones legales CONDUSEF relevantes para este caso
"""

    # ── Paso 4: Llamar a Claude API ──────────────────────────
    try:
        cliente_claude = obtener_claude_client()

        respuesta = cliente_claude.messages.create(
            model      = "claude-sonnet-4-20250514",
            max_tokens = 1000,
            messages   = [
                {
                    "role"   : "user",
                    "content": prompt_usuario
                }
            ],
            system = prompt_sistema
        )

        estrategia = respuesta.content[0].text

        return {
            'estrategia'    : estrategia,
            'query_rag'     : resultado_rag['query'],
            'docs_usados'   : [
                d['archivo']
                for d in resultado_rag['documentos']
            ],
            'tokens_usados' : respuesta.usage.input_tokens +
                              respuesta.usage.output_tokens,
            'exito'         : True
        }

    except Exception as e:
        return {
            'estrategia'  : f"Error al generar estrategia: {e}",
            'exito'       : False,
            'error'       : str(e)
        }


# ── Prueba con cliente real ───────────────────────────────────
print("=" * 60)
print("PRUEBA DE INTEGRACIÓN RAG + CLAUDE API")
print("=" * 60)

# Cliente de prueba con perfil realista
cliente_prueba = {
    'max_dias_atraso'     : 45,
    'probabilidad_ews'    : 0.78,
    'es_cliente_recurrente': 1,
    'carga_financiera'    : 0.42,
    'n_cuotas_total'      : 18,
    'max_dpd'             : 12,
}

# Top variables SHAP simuladas para este cliente
shap_prueba = [
    {'nombre': 'Máx. DPD mensual',      'shap': +0.85},
    {'nombre': '% cuotas atrasadas',     'shap': +0.43},
    {'nombre': 'Total de cuotas',        'shap': +0.38},
    {'nombre': 'Días última solicitud',  'shap': -0.21},
    {'nombre': '% pagos completos',      'shap': -0.18},
]

print(f"\nCliente de prueba:")
print(f"  Días de atraso  : {cliente_prueba['max_dias_atraso']}")
print(f"  Probabilidad EWS: {cliente_prueba['probabilidad_ews']:.0%}")
print(f"  Recurrente      : Sí")
print(f"\nGenerando estrategia con RAG + Claude API...")
print("(puede tardar 5-10 segundos)")

resultado = generar_estrategia_cobranza(
    cliente_info = cliente_prueba,
    shap_top     = shap_prueba,
    modo         = 'completo'
)

if resultado['exito']:
    print(f"\n{'='*60}")
    print("ESTRATEGIA GENERADA")
    print(f"{'='*60}")
    print(resultado['estrategia'])
    print(f"\n{'─'*60}")
    print(f"Documentos RAG usados : {resultado['docs_usados']}")
    print(f"Tokens consumidos     : {resultado['tokens_usados']}")
    print(f"\n✅ Integración RAG + Claude API funcionando")
else:
    print(f"\n❌ Error: {resultado['error']}")

PRUEBA DE INTEGRACIÓN RAG + CLAUDE API

Cliente de prueba:
  Días de atraso  : 45
  Probabilidad EWS: 78%
  Recurrente      : Sí

Generando estrategia con RAG + Claude API...
(puede tardar 5-10 segundos)

ESTRATEGIA GENERADA
## 🎯 ESTRATEGIA DE COBRANZA - CLIENTE RECURRENTE EN MORA MEDIA

### 1. ACCIÓN INMEDIATA (HOY)
**Gestión intensiva**: Realizar mínimo 2 intentos de contacto telefónico en horarios diferenciados (mañana y tarde). Como cliente recurrente con patrón de atrasos (máx. DPD 12 días vs actual 45), aplicar **gestión directa** sin extensiones adicionales.

### 2. GUIÓN SUGERIDO
- *"Buenos días [Nombre], le habla [Gestor] de [Empresa]. Veo que tiene 45 días de atraso, situación inusual considerando su buen historial previo."*
- *"Entiendo que pueden surgir dificultades. Tengo opciones de reestructura que pueden ayudarle a regularizar su cuenta hoy mismo."*
- *"¿Podemos revisar juntos cuál opción se adapta mejor a su situación actual?"*

### 3. OFERTA A PRESENTAR
**Reestructura